# KKBox — PostgreSQL Ingestion
### Loads 4 cleaned parquets into local PostgreSQL via COPY (bulk insert)

In [ ]:
import pandas as pd
import psycopg2
import io
import time
import os

CLEANED = r"./data/cleaned"

# Adjust credentials if needed
DB = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "kkbox",
    "user":     "postgres",
    "password": "password",
}

def get_conn():
    return psycopg2.connect(**DB)

# Test connection
try:
    conn = get_conn()
    conn.close()
    print("Connection successful")
except Exception as e:
    print(f"Connection failed: {e}")


## Step 1 — DDL: Create Tables

In [ ]:
DDL = """
DROP TABLE IF EXISTS transactions CASCADE;
DROP TABLE IF EXISTS members     CASCADE;
DROP TABLE IF EXISTS user_engagement CASCADE;
DROP TABLE IF EXISTS train        CASCADE;

CREATE TABLE transactions (
    user_id            VARCHAR(50),
    payment_method_id  SMALLINT,
    plan_duration_days SMALLINT,
    plan_price         SMALLINT,
    amount_paid        SMALLINT,
    is_auto_renew      SMALLINT,
    transaction_date   DATE,
    expire_date        DATE,
    is_cancel          SMALLINT,
    plan_type          VARCHAR(10),
    is_discounted      SMALLINT,
    is_payment_failed  SMALLINT,
    is_free_plan       SMALLINT
);

CREATE TABLE members (
    user_id              VARCHAR(50),
    city                 SMALLINT,
    gender               VARCHAR(10),
    registration_channel SMALLINT,
    registration_date    DATE
);

CREATE TABLE user_engagement (
    user_id           VARCHAR(50),
    avg_daily_songs   NUMERIC(8,2),
    total_listen_secs NUMERIC(12,2),
    active_days       SMALLINT,
    engagement_score  NUMERIC(6,4)
);

CREATE TABLE train (
    user_id  VARCHAR(50),
    is_churn SMALLINT
);
"""

conn = get_conn()
cur  = conn.cursor()
cur.execute(DDL)
conn.commit()
cur.close()
conn.close()
print("All tables created successfully")


## Step 2 — Bulk Load via COPY

In [ ]:
def bulk_load(df, table_name):
    """
    Streams a DataFrame into PostgreSQL using COPY.
    Writes to an in-memory CSV buffer — no temp files, no disk I/O.
    100-200x faster than to_sql() row-by-row insert.
    """
    conn = get_conn()
    cur  = conn.cursor()

    # Write DataFrame to in-memory CSV buffer
    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False, na_rep='')
    buffer.seek(0)

    # PostgreSQL COPY reads the buffer as if it were a file
    cur.copy_expert(
        f"COPY {table_name} FROM STDIN WITH (FORMAT CSV, NULL '')",
        buffer
    )

    conn.commit()
    cur.close()
    conn.close()


# Load order: independent tables first, transactions last
tables = [
    ("members",          "members_clean.parquet"),
    ("user_engagement",  "user_engagement.parquet"),
    ("train",            "train_clean.parquet"),
    ("transactions",     "transactions_clean.parquet"),
]

for table, fname in tables:
    start = time.time()
    df = pd.read_parquet(os.path.join(CLEANED, fname))

    # Flatten category dtype to int for PG SMALLINT columns
    for col in df.select_dtypes('category').columns:
        df[col] = df[col].astype(str) if df[col].cat.categories.dtype == object else df[col].astype('Int16')

    bulk_load(df, table)

    conn = get_conn()
    cur  = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    cur.close()
    conn.close()

    print(f"  {table:<20} {count:>10,} rows  ({time.time()-start:.1f}s)")


## Step 3 — Create Indexes

In [ ]:
INDEXES = [
    # user_id on every table
    "CREATE INDEX idx_txn_user_id      ON transactions    (user_id)",
    "CREATE INDEX idx_mem_user_id      ON members         (user_id)",
    "CREATE INDEX idx_eng_user_id      ON user_engagement (user_id)",
    "CREATE INDEX idx_train_user_id    ON train           (user_id)",

    # Composite index for window functions on (user_id, transaction_date)
    "CREATE INDEX idx_txn_user_date    ON transactions    (user_id, transaction_date)",

    # Additional indexes for GROUP BY / filter operations
    "CREATE INDEX idx_txn_date         ON transactions    (transaction_date)",
    "CREATE INDEX idx_txn_plan_type    ON transactions    (plan_type)",
    "CREATE INDEX idx_txn_is_cancel    ON transactions    (is_cancel)",
]

conn = get_conn()
cur  = conn.cursor()

for idx_sql in INDEXES:
    cur.execute(idx_sql)
    name = idx_sql.split("INDEX ")[1].split(" ")[0]
    print(f"  Created: {name}")

conn.commit()
cur.close()
conn.close()
print("\nAll indexes created")


## Step 4 — Verify

In [ ]:
checks = [
    ("Row counts",
     """SELECT 'transactions'   AS tbl, COUNT(*) FROM transactions
        UNION ALL
        SELECT 'members',                COUNT(*) FROM members
        UNION ALL
        SELECT 'user_engagement',        COUNT(*) FROM user_engagement
        UNION ALL
        SELECT 'train',                  COUNT(*) FROM train"""),

    ("transactions sample",
     """SELECT user_id, transaction_date, plan_type, plan_price,
               amount_paid, is_payment_failed, is_cancel
        FROM transactions LIMIT 3"""),

    ("members sample",
     "SELECT * FROM members LIMIT 3"),

    ("Date range",
     """SELECT MIN(transaction_date) AS min_date,
               MAX(transaction_date) AS max_date
        FROM transactions"""),

    ("Indexes on transactions",
     """SELECT indexname, indexdef
        FROM pg_indexes
        WHERE tablename = 'transactions'"""),
]

conn = get_conn()
cur  = conn.cursor()

for label, sql in checks:
    print(f"\n--- {label} ---")
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    df_out = pd.DataFrame(rows, columns=cols)
    display(df_out)

cur.close()
conn.close()
